# lazykh video renderer

Select **Runtime > Change runtime type > T4 GPU**, then run the cells in order. The upload cell prompts for one audio/video file, one annotated script, and optional billboard images. The final video is previewed and downloaded automatically.

The uploaded media may be `.wav`, `.mp3`, `.m4a`, `.mp4`, `.mov`, or `.webm`; it is converted to WAV automatically. Billboard images may be PNG, JPG, JPEG, or WEBP. Name each image for its bracketed script topic, such as `Ben.jpg` for `[Ben]`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/parubalo/lazykh.git"
BRANCH = "main"
WORKSPACE = Path("/content/lazykh")

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(WORKSPACE)], check=True)
os.chdir(WORKSPACE)

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)

In [ ]:
from google.colab import files

print("Upload one audio/video file, one annotated .txt script, and optional billboard images.")
uploaded = files.upload()
media_extensions = {".wav", ".mp3", ".m4a", ".mp4", ".mov", ".webm"}
image_extensions = {".png", ".jpg", ".jpeg", ".webp"}
media_names = [name for name in uploaded if Path(name).suffix.lower() in media_extensions]
script_names = [name for name in uploaded if Path(name).suffix.lower() == ".txt"]
if len(media_names) != 1 or len(script_names) != 1:
    raise ValueError("Upload exactly one supported media file and one .txt script.")

project_dir = Path("uploadedProject")
project_dir.mkdir(exist_ok=True)
media_path = project_dir / Path(media_names[0]).name
media_path.write_bytes(uploaded[media_names[0]])
script_path = project_dir / "input.txt"
script_path.write_bytes(uploaded[script_names[0]])
billboard_dir = project_dir / "input_billboards"
billboard_dir.mkdir(exist_ok=True)
billboard_names = [name for name in uploaded if Path(name).suffix.lower() in image_extensions]
billboard_paths = []
for name in billboard_names:
    source = Path(name)
    destination = billboard_dir / (source.stem.lower() + source.suffix.lower())
    destination.write_bytes(uploaded[name])
    billboard_paths.append(destination)

if billboard_paths:
    from IPython.display import display
    from PIL import Image
    print("Uploaded billboard images:")
    for path in billboard_paths:
        print(path.name)
        display(Image.open(path))
INPUT_FILE = str(project_dir / "input")
subprocess.run(["ffmpeg", "-y", "-i", str(media_path), "-vn", "-ac", "1", "-ar", "44100", "-c:a", "pcm_s16le", INPUT_FILE + ".wav"], check=True)
WHISPER_MODEL = "base"

In [ ]:
commands = [
    [sys.executable, "code/whisperAligner.py", "--input_file", INPUT_FILE, "--model", WHISPER_MODEL],
    [sys.executable, "code/scheduler.py", "--input_file", INPUT_FILE],
    [sys.executable, "code/videoDrawer.py", "--input_file", INPUT_FILE, "--use_billboards", "T", "--jiggly_transitions", "F"],
    [sys.executable, "code/videoFinisher.py", "--input_file", INPUT_FILE, "--keep_frames", "F", "--video_encoder", "auto"],
]

for command in commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

rendered_file = Path(INPUT_FILE + "_final.mp4")
if not rendered_file.is_file():
    raise FileNotFoundError(f"Rendering did not produce {rendered_file}")
OUTPUT_FILE = Path("/content/lazykh_output.mp4")
shutil.copy2(rendered_file, OUTPUT_FILE)

In [ ]:
from IPython.display import Video


Video("/content/lazykh_output.mp4", embed=True, width=600)


In [ ]:
from google.colab import files

files.download(str(OUTPUT_FILE))